# Behavior-trained Lee Kuan Yew simulation
Three roles: trained intent router, factual LoRA, and behavior LoRA sharing one base model. Behavior training spans leadership, crises, negotiation, technology, foreign relations and other domains. Select **Runtime > Change runtime type > T4 GPU**, then run cells in order. No API key or Drive mount is required for this public-repository workflow. This notebook is an unexecuted workflow template; executed local CPU results are in the repository. A GPU allocation is subject to your Colab quota.


In [ ]:
import os, pathlib, subprocess, sys
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training, or use the local CPU instructions.'
print(torch.cuda.get_device_name(0))
workspace = pathlib.Path('/content/99task_lee_kuan_yew_llm')
if not workspace.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DiedrickD/99group-3.git', str(workspace)], check=True)
os.chdir(workspace)
sys.path.insert(0, str(workspace))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)


## Choose a model
Keep 0.5B to reproduce the small experiment. Change to `Qwen/Qwen2.5-1.5B-Instruct` for a larger experiment; rerun preparation, training and evaluation together. Do not compare results across model sizes as if only the adapter changed. The exact model commit is recorded automatically.


In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
def run(*args):
    subprocess.run([sys.executable, *args], check=True)
run('scripts/train_router.py')
run('scripts/build_behavior_dataset.py')
run('scripts/build_broad_behavior_dataset.py')
run('scripts/validate_data.py')
run('scripts/prepare_model.py', '--model-id', MODEL_ID)
run('-m', 'lky.train', '--epochs', '2')


## Train the behavior adapter
Stage 1: 24 event/action/analogy examples, three epochs. Stage 2: 32 broader examples plus eight replayed examples, one epoch. Both stages use the same base revision as the newly trained factual adapter. Synthetic applications are explicitly labelled interpretations, not actual views or private thought transcripts. The two validation examples come from a source absent from these training targets.


In [ ]:
run('-m', 'lky.train', '--epochs', '3', '--max-length', '512', '--train-data', 'data/behavior_train.jsonl', '--validation-data', 'data/behavior_validation.jsonl', '--report', 'reports/colab-behavior-stage1.json', '--resume-adapter', 'models/lky-lora', '--output', 'models/checkpoints/behavior-stage1')
run('-m', 'lky.train', '--epochs', '1', '--max-length', '512', '--train-data', 'data/behavior_broad_train.jsonl', '--validation-data', 'data/behavior_validation.jsonl', '--report', 'reports/colab-behavior-training.json', '--resume-adapter', 'models/checkpoints/behavior-stage1', '--output', 'models/lky-behavior')


## Evaluate
The 16 development cases check routing, labels, analogy retrieval and response structure. The same four-case comparison switches the behavior adapter off while keeping the prompt and evidence. Inspect raw output for relevance and fidelity: structure is not an accuracy score. These runs write separate Colab reports rather than replacing the recorded local v3 results.


In [ ]:
run('-m', 'lky.evaluate_persona', '--backend', 'notes', '--output', 'reports/colab-behavior-notes.json')
run('-m', 'lky.evaluate_persona', '--backend', 'lora', '--original-adapter', '--ids', 'chatgpt-false-premise,earthquake,leadership,economic-crisis', '--output', 'reports/colab-behavior-baseline.json')
run('-m', 'lky.evaluate_persona', '--backend', 'lora', '--output', 'reports/colab-behavior-lora.json')


In [ ]:
# Download both trained adapters, the router and measured reports before expiry.
# Published manifests describe the committed local run, not these new weights.
import zipfile
from google.colab import files
archive = '/content/lky-training-artifacts.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as output:
    for folder in ['models/lky-lora', 'models/lky-behavior', 'reports', 'config']:
        for path in pathlib.Path(folder).rglob('*'):
            if path.is_file():
                output.write(path, path.as_posix())
    output.write('models/router.json', 'models/router.json')
files.download(archive)


## Temporary chat demo
The next cell creates a public Gradio share URL. It works only while this runtime and server remain available. Use the repository chat examples as the permanent submission link. Each turn is independent.


In [ ]:
from app import build_app
demo = build_app('lora')
demo.launch(share=True, debug=False)
